# Importing required packages

In [6]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import cellcharter as cc
import matplotlib.pyplot as plt
import yaml
import squidpy as sq
from pathlib import Path

# Required inputs

In [7]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set up directories
metadata_dir = Path(config['metadata_dir'])
preprocessed_dir = Path(config['preprocessed_dir'])

# make a roi list of directories in preprocessed_dir that are not .DS_Store
roi_list = [roi for roi in preprocessed_dir.iterdir() if roi.is_dir() and roi.name != '.DS_Store']

# Adata concatenation

In [8]:
#conctenate all the adata files
adata_list = []
for roi in roi_list:
    adata = ad.read_h5ad(roi / "adata.h5ad")
    adata_list.append(adata)
adata = ad.concat(adata_list)

#make obs unique in adata
adata.obs_names_make_unique()

adata.obs["exp_name"] = adata.obs["exp_name"].astype("category")
adata.obs["patient_ID"] = adata.obs["patient_ID"].astype("category")

/opt/anaconda3/envs/cellcharter_env/lib/python3.9/site-packages/anndata/_core/merge.py:1284: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/opt/anaconda3/envs/cellcharter_env/lib/python3.9/site-packages/anndata/_core/merge.py:1284: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/opt/anaconda3/envs/cellcharter_env/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.

# Adata scaling

In [9]:
#log transform adata
sc.pp.scale(adata)
adata.X = adata.X.astype(np.float32).copy()

# Training TRVAE

In [5]:
#Training my own model 
condition_key = 'patient_ID'
cell_type_key = 'cell_type'
conditions = adata.obs[condition_key].unique().tolist()


trvae_epochs = 500
surgery_epochs = 500

early_stopping_kwargs = {
    "early_stopping_metric": "val_unweighted_loss",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
}
trvae = cc.tl.TRVAE(
    adata=adata,
    condition_key=condition_key,
    conditions=conditions,
    recon_loss='mse',
    use_mmd=False,
)
trvae.train(
    n_epochs=trvae_epochs,
    alpha_epoch_anneal=200,
    early_stopping_kwargs=early_stopping_kwargs,
    enable_progress_bar=True,
    
)
trvae.save('trvae', overwrite=True)


INITIALIZING NEW NETWORK..............
Encoder Architecture:
	Input Layer in, out and cond: 61 256 14
	Hidden Layer 1 in/out: 256 64
	Mean/Var Layer in/out: 64 10
Decoder Architecture:
	First Layer in, out and cond:  10 64 14
	Hidden Layer 1 in/out: 64 256
	Output Layer in/out:  256 61 

Preparing (309273, 61)
Instantiating dataset
 |█████---------------| 25.2%  - val_loss: 18.0111656859 - val_recon_loss: 14.0321877535 - val_kl_loss: 6.36636483571
ADJUSTED LR
 |██████--------------| 34.6%  - val_loss: 17.3962421693 - val_recon_loss: 12.8854625304 - val_kl_loss: 5.2450925417
ADJUSTED LR
 |████████------------| 40.4%  - val_loss: 17.9181165459 - val_recon_loss: 13.0535643377 - val_kl_loss: 4.8645521511
ADJUSTED LR
 |████████------------| 44.4%  - val_loss: 17.8647209554 - val_recon_loss: 13.0006439095 - val_kl_loss: 4.8640769966
ADJUSTED LR
 |██████████----------| 53.2%  - val_loss: 17.8672083350 - val_recon_loss: 12.9976128015 - val_kl_loss: 4.8695954843
ADJUSTED LR
 |██████████-------